# Layered Control Architecture for 6-Agent Escort Formation ($SE(3)$ Homogeneous Sheaf)

This example demonstrates scaling the layered control architecture to a 6-agent formation
escorting a slow-moving target in 3D space using an **$SE(3)$ Homogeneous Affine Cellular Sheaf**.

## High-Level API

We use the new `Formations` and `DistributedLayeredControl` modules to significantly
simplify the setup of the escort ring and the execution of the distributed local controllers.

In [1]:
using CellularSheaves
using CellularSheaves.Formations
using CellularSheaves.AgentControllers
using CellularSheaves.DistributedLayeredControl
using LinearAlgebra
using Statistics
using Distributed
using Plots
using Printf

A single house style for every figure below

In [2]:
default(framestyle = :box, grid = true, gridalpha = 0.18, gridstyle = :dot,
    titlefontsize = 10, guidefontsize = 9, legendfontsize = 8, tickfontsize = 8,
    markerstrokewidth = 0, size = (720, 380))

const RING_COLOR = :steelblue
const TARGET_COLOR = :black

:black

## Setup 10D Quadrotor Dynamics & DARE Solver

In [3]:
dyn = QuadrotorDynamics()
DT = 0.05
nx = 10

10

Compute Optimal LQR Gain via Discrete Algebraic Riccati Equation (DARE)

In [4]:
Q_diag = [500.0, 500.0, 500.0, 150.0, 150.0, 100.0, 100.0, 100.0, 5.0, 5.0]
Q_lqr = Matrix(Diagonal(Q_diag))
R_lqr = Matrix(Diagonal([0.005, 0.005, 0.005]))

lqr_controller = LQRController(dyn, DT, Q_lqr, R_lqr)
K_lqr = lqr_controller.K

3×10 Matrix{Float64}:
 0.0       0.0      21.0723  0.0      0.0      …  10.4824  0.0       0.0
 0.0      -1.46667   0.0     2.39227  0.0          0.0     0.255503  0.0
 1.46667   0.0       0.0     0.0      2.39227      0.0     0.0       0.255503

## SE(3) Homogeneous Coordination Sheaf Construction (D = 4)

6 agents in a single escort ring.

In [5]:
const NA = 6
const NT = 1
const TV1 = NA + 1
r_ring = 0.3

0.3

Build the escort ring, with Agent 1 pinned as the observer

In [6]:
sheaf = build_escort_ring(NA, TV1, r_ring; observers=[1])

A network sheaf with 7 vertex stalks and 7 edge stalks.


Fast-moving target trajectory

In [7]:
target1_pos(node, t) = [0.5cos(0.5*t), 0.5sin(0.5*t), 1.5 + 0.1sin(1.0*t), 1.0]

target1_pos (generic function with 1 method)

## Provision Worker Processes

Provision exactly NA worker processes (one for each agent's flight computer)

In [8]:
workers_pids = addprocs(NA; exeflags = "--project=$(Base.active_project())")

6-element Vector{Int64}:
  5
  6
  7
  8
  9
 10

Provide the cellular sheaves environment to all workers

In [9]:
@everywhere workers_pids begin
    using CellularSheaves
end

## Simulation Framework

In [10]:
STEPS = 200
epsilon = 0.02

0.02

Start agents in a line along the x-axis

In [11]:
init_states = [[r_ring*i/NA, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] for i in 1:NA]

6-element Vector{Vector{Float64}}:
 [0.049999999999999996, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
 [0.09999999999999999, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
 [0.15, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
 [0.19999999999999998, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
 [0.25, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
 [0.3, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

Create heterogeneous agent properties (varying mass and inertia)

In [12]:
dyns = [QuadrotorDynamics(m=0.5 + 0.05*i, Ixx=0.01 + 0.002*i, Iyy=0.01 + 0.002*i) for i in 1:NA]
K_lqrs = [LQRController(d, DT, Q_lqr, R_lqr).K for d in dyns]
agent_configs = [(init_states[i], dyns[i], K_lqrs[i]) for i in 1:NA]

prob = LayeredControlProblem(sheaf, [TV1], target1_pos, agent_configs, DT, STEPS, r_ring)

CellularSheaves.ControlSheaves.DistributedLayeredControl.LayeredControlProblem(A network sheaf with 7 vertex stalks and 7 edge stalks.
, [7], Main.var"##791".target1_pos, nothing, nothing, Tuple{Vector{Float64}, CellularSheaves.ControlSheaves.AgentControllers.QuadrotorDynamics, Matrix{Float64}}[([0.049999999999999996, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], CellularSheaves.ControlSheaves.AgentControllers.QuadrotorDynamics(9.81, 0.55, 0.012, 0.012), [0.0 0.0 … 0.0 0.0; 0.0 -1.7599735965077932 … 0.30659989356367856 0.0; 1.7599735965077927 0.0 … 0.0 0.30659989356367856]), ([0.09999999999999999, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], CellularSheaves.ControlSheaves.AgentControllers.QuadrotorDynamics(9.81, 0.6, 0.014, 0.014), [0.0 0.0 … 0.0 0.0; 0.0 -2.0532605811338844 … 0.3576954723740814 0.0; 2.0532605811338835 0.0 … 0.0 0.3576954723740814]), ([0.15, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], CellularSheaves.ControlSheaves.AgentControllers.QuadrotorDynamics(9.81, 0.65, 0.01

Run distributed simulation

In [13]:
init_distributed_agents!(workers_pids, agent_configs, DT, epsilon)
sim_d_res = run_layered_simulation(prob, workers_pids; mode=:distributed)

CellularSheaves.ControlSheaves.DistributedLayeredControl.LayeredSimulationResult(CellularSheaves.ControlSheaves.DistributedLayeredControl.LayeredControlProblem(A network sheaf with 7 vertex stalks and 7 edge stalks.
, [7], Main.var"##791".target1_pos, nothing, nothing, Tuple{Vector{Float64}, CellularSheaves.ControlSheaves.AgentControllers.QuadrotorDynamics, Matrix{Float64}}[([0.049999999999999996, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], CellularSheaves.ControlSheaves.AgentControllers.QuadrotorDynamics(9.81, 0.55, 0.012, 0.012), [0.0 0.0 … 0.0 0.0; 0.0 -1.7599735965077932 … 0.30659989356367856 0.0; 1.7599735965077927 0.0 … 0.0 0.30659989356367856]), ([0.09999999999999999, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], CellularSheaves.ControlSheaves.AgentControllers.QuadrotorDynamics(9.81, 0.6, 0.014, 0.014), [0.0 0.0 … 0.0 0.0; 0.0 -2.0532605811338844 … 0.3576954723740814 0.0; 2.0532605811338835 0.0 … 0.0 0.3576954723740814]), ([0.15, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], C

Run centralized simulation

In [14]:
init_distributed_agents!(workers_pids, agent_configs, DT, epsilon)
sim_c_res = run_layered_simulation(prob, workers_pids; mode=:centralised)

divergence = maximum(abs.(sim_d_res.sim_data .- sim_c_res.sim_data))
@printf("Max divergence between centralized and distributed 6-agent simulation: %.3e\n", divergence)

Max divergence between centralized and distributed 6-agent simulation: 4.752e-14


Clean up worker processes

In [15]:
rmprocs(workers_pids)

Task (done) @0x00007f68b9738d30

## Multi-Projection Trajectory & Attitude Dynamics Visualization

We can now easily animate the simulation using the Plots.jl recipe framework.
The `animate_layered_escort` function abstracts away the complex multi-panel visualization.
(Note: this relies on the `CellularSheavesPlots` package extension)

In [16]:
animate_layered_escort(sim_d_res; frame_step = 2, filename = "layered_escort_tracking.gif", fps = 15)
nothing # hide

[ Info: Saved animation to /home/runner/work/CellularSheaves.jl/CellularSheaves.jl/docs/src/generated/layered/layered_escort_tracking.gif


![6-Agent Escort Tracking Projections](layered_escort_tracking.gif)